In [ ]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
data_path = project_root / "data" / "CEAS_08.csv"

if not data_path.exists():
    data_path = project_root.parent / "data" / "CEAS_08.csv"

df = pd.read_csv(data_path)

39157 email

7 thuộc tính

In [ ]:
print(df.shape)

print(df.columns)

df.info()

df["label"].value_counts()

In [ ]:
df.head()


In [ ]:
# Vẽ biểu đồ nhãn
import matplotlib.pyplot as plt

df["label"].value_counts().plot(kind="bar")

plt.title("Distribution of Labels")

plt.show()


In [ ]:
#tạo dữ liệu văn bản đầu vào
df["subject"] = df["subject"].fillna("")

df["text"] = df["subject"] + " " + df["body"]

In [ ]:
df[["subject","body","text"]].head()

In [ ]:
import re

def clean_text(text):

    text = str(text)

    # chuyển về chữ thường
    text = text.lower()

    # xóa url
    text = re.sub(r'http\S+', ' ', text)

    # chỉ giữ chữ cái tiếng Anh
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # xóa khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

df["clean_text"] = df["text"].apply(clean_text)

In [ ]:
#kiểm tra kết quả làm sạch dữ liệu văn bản
df[["text","clean_text"]].head()

In [ ]:
#remove stopwords
import nltk

nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
def remove_stopwords(text):

    words = text.split()

    filtered_words = [
        word
        for word in words
        if word not in stop_words
    ]

    return " ".join(filtered_words)
df["clean_text"] = df["clean_text"].apply(remove_stopwords)

In [ ]:
#check
df[["text","clean_text"]].head()

In [ ]:
#TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    max_features=5000
)
X = tfidf.fit_transform(df["clean_text"])


In [ ]:
#check shape
print(X.shape)

39154 email

5000 từ khóa quan trọng nhất

In [ ]:
#check feature names
print(tfidf.get_feature_names_out()[:100])

In [ ]:
# Train/Test Split
#Chia dư liệu thành tập huấn luyện và tập kiểm tra
from sklearn.model_selection import train_test_split
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
#check
print(X_train.shape)
print(X_test.shape)
#ktra phân phối nhãn 
print(y_train.value_counts())
print(y_test.value_counts())

31323 email dùng để train
7831 email dùng để test => 80% dữ liệu để huấn luyện
20% dữ liệu để kiểm tra

Tỷ lệ:
Train:
1 ≈ 55.9%
0 ≈ 44.1%
Test:
1 ≈ 55.4%
0 ≈ 44.6%

In [ ]:
#Huấn luyện mô hình Random Forest
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
#rf_model.fit(X_train, y_train)
#print("Training completed!")

In [ ]:
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

In [ ]:
#Đánh giá
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

In [ ]:
#Vẽ Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)

print(cm)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm
)

disp.plot()

plt.show()

In [ ]:
#Feature Importance
feature_importances = rf_model.feature_importances_
print(len(feature_importances))
max_features=5000

In [ ]:
feature_names = tfidf.get_feature_names_out()
print(len(feature_names))

In [ ]:
import pandas as pd

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": feature_importances
})
importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)
importance_df.head(20)

In [ ]:
top20 = importance_df.head(20)
import matplotlib.pyplot as plt
plt.figure(figsize=(12,6))

plt.barh(
    top20["Feature"],
    top20["Importance"]
)

plt.xlabel("Importance")

plt.ylabel("Feature")

plt.title(
    "Top 20 Important Features in Random Forest"
)

plt.gca().invert_yaxis()

plt.show()

In [ ]:
#Lưu mô hình
import joblib
joblib.dump(
    rf_model,
    "../model/rf_model.pkl"
)
joblib.dump(
    tfidf,
    "../model/tfidf.pkl"
)

In [ ]:
import os

print(os.listdir("../model"))

In [ ]:
import os

print(os.path.abspath("../model"))

In [ ]:
import os

print(
    os.path.exists("../model/rf_model.pkl")
)

print(
    os.path.exists("../model/tfidf.pkl")
)